<a href="https://colab.research.google.com/github/krvivek5/AlignIQ-Backend-Experiment/blob/main/job_matching_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import json
import pandas as pd
import numpy as np
from typing import Dict, List, Any, Tuple
import spacy
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import joblib

In [2]:
# Load NLP model
try:
    nlp = spacy.load("en_core_web_lg")
except:
    # If model isn't available, download it
    import subprocess
    subprocess.call(["python", "-m", "spacy", "download", "en_core_web_lg"])
    nlp = spacy.load("en_core_web_lg")

In [3]:
class JobMatchingPipeline:
    def __init__(self, skills_taxonomy_path: str = None):
        """
        Initialize the job matching pipeline

        Args:
            skills_taxonomy_path: Path to skills taxonomy JSON file
        """
        self.skills_taxonomy = self._load_skills_taxonomy(skills_taxonomy_path)
        self.vectorizer = TfidfVectorizer(stop_words='english')
        self.skills_patterns = self._compile_skills_regex()
        self.education_patterns = self._compile_education_regex()
        self.experience_patterns = self._compile_experience_regex()

    def _load_skills_taxonomy(self, path: str) -> Dict:
        """Load skills taxonomy from file or use default"""
        if path and os.path.exists(path):
            with open(path, 'r') as f:
                return json.load(f)
        else:
            # Default minimal taxonomy - in production, use a comprehensive one
            return {
                "technical_skills": [
                    "python", "java", "javascript", "sql", "aws", "docker",
                    "kubernetes", "react", "angular", "vue", "node.js", "express",
                    "django", "flask", "tensorflow", "pytorch", "machine learning",
                    "data science", "data analysis", "data engineering", "devops"
                ],
                "soft_skills": [
                    "communication", "teamwork", "leadership", "problem solving",
                    "time management", "critical thinking", "adaptability",
                    "creativity", "project management", "conflict resolution"
                ]
            }

    def _compile_skills_regex(self) -> Dict[str, re.Pattern]:
        """Compile regex patterns for skills extraction"""
        patterns = {}
        all_skills = (
            self.skills_taxonomy.get("technical_skills", []) +
            self.skills_taxonomy.get("soft_skills", [])
        )

        # Create pattern that matches skills with word boundaries
        skill_pattern = r'\b(?:' + '|'.join(re.escape(skill) for skill in all_skills) + r')\b'
        patterns['skills'] = re.compile(skill_pattern, re.IGNORECASE)

        return patterns

    def _compile_education_regex(self) -> Dict[str, re.Pattern]:
        """Compile regex patterns for education extraction"""
        patterns = {}

        # Degrees pattern
        degrees = [
            "Bachelor", "BS", "BA", "B.S.", "B.A.",
            "Master", "MS", "MA", "M.S.", "M.A.", "MBA",
            "PhD", "Ph.D.", "Doctorate", "Associate", "Certificate"
        ]
        degree_pattern = r'\b(?:' + '|'.join(re.escape(deg) for deg in degrees) + r')\b'
        patterns['degrees'] = re.compile(degree_pattern, re.IGNORECASE)

        # Fields of study - simplified for example
        fields = [
            "Computer Science", "Information Technology", "Engineering",
            "Business", "Marketing", "Data Science", "Mathematics"
        ]
        field_pattern = r'\b(?:' + '|'.join(re.escape(field) for field in fields) + r')\b'
        patterns['fields'] = re.compile(field_pattern, re.IGNORECASE)

        return patterns

    def _compile_experience_regex(self) -> Dict[str, re.Pattern]:
        """Compile regex patterns for experience extraction"""
        patterns = {}

        # Years of experience pattern
        patterns['years'] = re.compile(
            r'\b(\d+)(?:\+)?\s*(?:years?|yrs?)(?:\s+of)?\s+(?:experience|exp)\b',
            re.IGNORECASE
        )

        # Job titles - simplified for example
        job_titles = [
            "Software Engineer", "Developer", "Data Scientist", "Project Manager",
            "Product Manager", "Designer", "Analyst", "Engineer", "Director",
            "Manager", "Lead", "Architect", "Administrator", "DevOps"
        ]
        title_pattern = r'\b(?:' + '|'.join(re.escape(title) for title in job_titles) + r')\b'
        patterns['titles'] = re.compile(title_pattern, re.IGNORECASE)

        return patterns

    def extract_data_from_chat(self, chat_text: str) -> Dict[str, Any]:
        """
        Extract job seeker data from chat text

        Args:
            chat_text: The text of the chat conversation

        Returns:
            Dictionary containing extracted data
        """
        # Process text with spaCy
        doc = nlp(chat_text)

        # Initialize results dictionary
        data = {
            "skills": [],
            "education": {
                "degrees": [],
                "fields": []
            },
            "experience": {
                "years": None,
                "previous_roles": []
            },
            "preferences": {
                "salary": None,
                "location": [],
                "remote": None
            }
        }

        # Extract skills
        skill_matches = self.skills_patterns['skills'].findall(chat_text)
        data["skills"] = list(set([match.lower() for match in skill_matches]))

        # Extract education
        degree_matches = self.education_patterns['degrees'].findall(chat_text)
        data["education"]["degrees"] = list(set([match for match in degree_matches]))

        field_matches = self.education_patterns['fields'].findall(chat_text)
        data["education"]["fields"] = list(set([match for match in field_matches]))

        # Extract experience
        year_matches = self.experience_patterns['years'].findall(chat_text)
        if year_matches:
            # Take the highest number of years mentioned
            data["experience"]["years"] = max([int(year) for year in year_matches])

        title_matches = self.experience_patterns['titles'].findall(chat_text)
        data["experience"]["previous_roles"] = list(set([match for match in title_matches]))

        # Extract salary expectations (simplified approach)
        salary_pattern = r'\$?\s*(\d{1,3}(?:,\d{3})*|\d+)(?:k|K)?\s*(?:-|to|–)?\s*\$?\s*(\d{1,3}(?:,\d{3})*|\d+)?(?:k|K)?'
        salary_matches = re.findall(salary_pattern, chat_text)
        if salary_matches:
            # Process the first match
            match = salary_matches[0]
            # Convert to consistent format
            min_salary = match[0].replace(',', '')
            min_salary = int(min_salary) * 1000 if min_salary.endswith(('k', 'K')) else int(min_salary)

            if match[1]:  # If there's a range
                max_salary = match[1].replace(',', '')
                max_salary = int(max_salary) * 1000 if max_salary.endswith(('k', 'K')) else int(max_salary)
                data["preferences"]["salary"] = (min_salary, max_salary)
            else:
                data["preferences"]["salary"] = min_salary

        # Extract location preferences using entity recognition
        for ent in doc.ents:
            if ent.label_ == "GPE":  # Geographical entity
                data["preferences"]["location"].append(ent.text)

        # Check for remote preferences
        remote_pattern = r'\b(remote|work from home|wfh|telework|virtual)\b'
        remote_matches = re.findall(remote_pattern, chat_text, re.IGNORECASE)
        if remote_matches:
            data["preferences"]["remote"] = True

        return data

    def match_jobs(self, candidate_data: Dict[str, Any], job_listings: List[Dict]) -> List[Tuple[Dict, float]]:
        """
        Match candidate with suitable jobs

        Args:
            candidate_data: Extracted candidate data
            job_listings: List of job listings

        Returns:
            List of (job, score) tuples, sorted by match score
        """
        # Prepare candidate profile
        candidate_skills = ' '.join(candidate_data["skills"])
        candidate_years = candidate_data["experience"]["years"] or 0

        matched_jobs = []

        for job in job_listings:
            # Calculate skills match score
            job_skills = ' '.join(job.get("required_skills", []) + job.get("preferred_skills", []))

            # If either is empty, we'll get errors with TF-IDF, so handle this case
            if not candidate_skills or not job_skills:
                skill_score = 0
            else:
                # Vectorize skills
                skills_matrix = self.vectorizer.fit_transform([candidate_skills, job_skills])
                skill_score = cosine_similarity(skills_matrix[0:1], skills_matrix[1:2])[0][0]

            # Calculate experience match
            required_years = job.get("required_experience", 0)
            if candidate_years >= required_years:
                exp_score = 1.0
            else:
                # Partial credit for close matches
                exp_score = max(0, 1 - (required_years - candidate_years) / required_years)

            # Calculate location match
            location_score = 0
            if job.get("location") in candidate_data["preferences"]["location"]:
                location_score = 1.0
            elif job.get("remote", False) and candidate_data["preferences"]["remote"]:
                location_score = 0.8  # Good but not perfect match

            # Calculate overall score (weighted average)
            overall_score = 0.5 * skill_score + 0.3 * exp_score + 0.2 * location_score

            matched_jobs.append((job, overall_score))

        # Sort by score descending
        return sorted(matched_jobs, key=lambda x: x[1], reverse=True)

    def save_model(self, path: str = "job_matching_model.pkl"):
        """Save the model for later use"""
        model_data = {
            "vectorizer": self.vectorizer,
            "skills_taxonomy": self.skills_taxonomy,
            "skills_patterns": self.skills_patterns,
            "education_patterns": self.education_patterns,
            "experience_patterns": self.experience_patterns
        }
        joblib.dump(model_data, path)

    @classmethod
    def load_model(cls, path: str = "job_matching_model.pkl"):
        """Load a saved model"""
        model_data = joblib.load(path)

        pipeline = cls()  # Create empty pipeline
        pipeline.vectorizer = model_data["vectorizer"]
        pipeline.skills_taxonomy = model_data["skills_taxonomy"]
        pipeline.skills_patterns = model_data["skills_patterns"]
        pipeline.education_patterns = model_data["education_patterns"]
        pipeline.experience_patterns = model_data["experience_patterns"]

        return pipeline

In [4]:
# Example usage
if __name__ == "__main__":
    # Sample chat text
    chat_text = """
    User: I'm looking for a new job in software engineering.
    Bot: Great! Can you tell me about your skills?
    User: I have 5 years of experience with Python and JavaScript, and I've used React for 3 years.
    Bot: That's impressive! What about your education?
    User: I have a Bachelor's in Computer Science from UCLA.
    Bot: Any specific job requirements?
    User: I'd prefer a remote position with a salary of at least $120k. I'm interested in mid-sized tech companies.
    Bot: When would you be available to start?
    User: I can start in two weeks.
    Bot: Past company name ?
    User: I also worked in startup called KavNex AI
    Bot: what is your previous role ?
    User: System Engineer.
    """

    # Sample job listings
    job_listings = [
        {
            "title": "Senior Software Engineer",
            "company": "TechCorp",
            "location": "San Francisco",
            "remote": True,
            "required_skills": ["python", "javascript", "react"],
            "preferred_skills": ["aws", "docker"],
            "required_experience": 3,
            "salary_range": (110000, 140000)
        },
        {
            "title": "Full Stack Developer",
            "company": "Startup Inc",
            "location": "New York",
            "remote": False,
            "required_skills": ["javascript", "node.js", "react"],
            "preferred_skills": ["python", "typescript"],
            "required_experience": 2,
            "salary_range": (90000, 120000)
        },
        {
            "title": "Backend Engineer",
            "company": "Big Enterprise",
            "location": "Austin",
            "remote": True,
            "required_skills": ["java", "spring", "sql"],
            "preferred_skills": ["kubernetes", "docker"],
            "required_experience": 5,
            "salary_range": (130000, 160000)
        }
    ]

    # Create and use pipeline
    pipeline = JobMatchingPipeline()
    candidate_data = pipeline.extract_data_from_chat(chat_text)

    print("Extracted candidate data:")
    print(json.dumps(candidate_data, indent=2))

    matches = pipeline.match_jobs(candidate_data, job_listings)

    print("\nTop job matches:")
    for job, score in matches:
        print(f"{job['title']} at {job['company']} - Match score: {score:.2f}")

Extracted candidate data:
{
  "skills": [
    "react",
    "python",
    "javascript"
  ],
  "education": {
    "degrees": [
      "Bachelor"
    ],
    "fields": [
      "engineering",
      "Computer Science"
    ]
  },
  "experience": {
    "years": 5,
    "previous_roles": [
      "Engineer"
    ]
  },
  "preferences": {
    "salary": 5,
    "location": [],
    "remote": true
  }
}

Top job matches:
Senior Software Engineer at TechCorp - Match score: 0.79
Full Stack Developer at Startup Inc - Match score: 0.59
Backend Engineer at Big Enterprise - Match score: 0.46
